# TS-AgentBench: a walkthrough

This notebook builds a small counterfactual tensor from scratch and uses it to answer
the question the benchmark exists for: **was correcting the right call?**

It runs in a couple of minutes on a CPU and needs no API key.

Run from the repository root:
```bash
jupyter notebook notebooks/01_benchmark_walkthrough.ipynb
```

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.simplefilter('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets.synthetic import generate_family, family_names
from datasets.base import make_split, iter_instances

print('generator families:', len(family_names()))
print(family_names())

## 1. Synthetic series carry ground truth

Real data cannot tell us where the "true" change point is. Synthetic series can, and
that is what makes the causal analysis (H2, H3) possible.

In [ ]:
s = generate_family('level_shift', 1, base_seed=0, n_timesteps=320, seasonal_period=12)[0]
cp = s.labels.changepoints[0]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(s.target, lw=0.9, color='black')
ax.axvline(cp, color='crimson', ls='--', label=f'true change point (t={cp})')
ax.set_title(f'{s.series_id}  --  expected failure mode: '
             f"{[m.value for m in s.labels.expected_failure_modes]}")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. Diagnostics see history only

Every tool takes `history` and is never given the future. That constraint is enforced
by the signature and tested in `tests/test_leakage.py`.

In [ ]:
from tools.base import TOOLS

origin = cp + 30            # forecast from just after the break
history = s.target[:origin]

for name in ['changepoint', 'anomaly', 'stationarity', 'distribution_shift', 'stl']:
    r = TOOLS.build(name).run(history, s.seasonal_period)
    print(f'{name:20s} {r.summary}')

detected = TOOLS.build('changepoint').run(history, s.seasonal_period).get('changepoints')
print(f'\ntrue change point: {cp}   detected: {detected}')

## 3. Build a counterfactual tensor

For each instance we run **every** corrective action and score each one. This is what
no other time-series benchmark records, and it is what makes the correction *decision*
measurable rather than only its downstream effect.

In [ ]:
from benchmark.tensor import build_tensor

MODELS = ['naive', 'seasonal_naive', 'drift', 'theta', 'ets']
FAMILIES = ['level_shift', 'trend_damped', 'anomaly_contaminated', 'seasonal_single']

series = [x for f in FAMILIES
          for x in generate_family(f, 4, base_seed=0, n_timesteps=240, seasonal_period=12)]

instances = {}
for x in series:
    sp = make_split(len(x), val_length=36, test_length=36, min_train_length=32)
    instances[x.series_id] = list(
        iter_instances(x, sp, horizons=[1, 12], n_val_origins=3,
                       n_test_origins=2, origin_stride=8))

outcomes, signals = build_tensor(series, instances, MODELS, base_seed=0,
                                 n_jobs=-1, progress=False)
print(f'outcomes: {outcomes.shape}   signals: {signals.shape}')
outcomes.head()

## 4. Do corrections actually help?

This is the question the literature assumes the answer to.

In [ ]:
test = outcomes[outcomes.split == 'test'].copy()
test.loc[~test.applicable.astype(bool), 'mase'] = np.nan
accept = test[test.action == 'accept'].set_index('instance_id')['mase']

rows = []
for action in test.action.unique():
    sub = test[test.action == action].set_index('instance_id')['mase']
    common = accept.index.intersection(sub.dropna().index)
    if len(common) == 0:
        continue
    rows.append({
        'action': action,
        'applicable': test[test.action == action].applicable.mean(),
        'beats_accept': (sub[common] < accept[common] - 1e-12).mean(),
        'median_delta_mase': float(np.median(sub[common] - accept[common])),
    })
pd.DataFrame(rows).sort_values('beats_accept', ascending=False).round(3)

Note how often a repair makes things **worse**. Correction is not free, and applying
one unconditionally is a bet rather than an improvement.

## 5. Two oracles, and why the difference matters

The unrestricted oracle takes the minimum over ten noisy outcomes, so part of its
apparent advantage is the winner's curse. The realizable oracle chooses on validation
and is attainable.

In [ ]:
from benchmark.oracle import add_oracle_labels, add_realizable_oracle

labels = add_oracle_labels(outcomes[outcomes.split == 'test'], delta=0.10)
realizable = add_realizable_oracle(outcomes, metric='mase')
m = labels.merge(realizable, on='instance_id')

print(f"accept              MASE = {m.accept_metric.mean():.4f}")
print(f"realizable oracle   MASE = {m.realizable_metric.mean():.4f}   (attainable)")
print(f"unrestricted oracle MASE = {m.oracle_metric.mean():.4f}   (selection-inflated)")
print(f"\nshould_correct rate = {labels.should_correct.mean():.1%}")
print(f"mean fraction of actions that would HARM = {labels.harmful_fraction.mean():.1%}")

## 6. Does validation evidence transfer to test?

This is the premise the whole method rests on. If an action's benefit on recent history
says nothing about its benefit on the next window, no gating policy can work.

In [ ]:
from scipy import stats

def gains(split):
    d = outcomes[outcomes.split == split].copy()
    d.loc[~d.applicable.astype(bool), 'mase'] = np.nan
    acc = d[d.action == 'accept'].groupby(['series_id', 'horizon'])['mase'].mean().rename('accept')
    act = d[d.action != 'accept'].groupby(['series_id', 'horizon', 'action'])['mase'].mean().rename('m')
    g = act.reset_index().merge(acc.reset_index(), on=['series_id', 'horizon'])
    g['gain'] = (g['accept'] - g['m']) / g['accept'].replace(0, np.nan)
    return g

v = gains('val').rename(columns={'gain': 'val_gain'})
t = gains('test').rename(columns={'gain': 'test_gain'})
j = v.merge(t, on=['series_id', 'horizon', 'action']).dropna(subset=['val_gain', 'test_gain'])

r, p = stats.pearsonr(j.val_gain, j.test_gain)
print(f'validation gain vs test gain:  r = {r:+.3f}  (p = {p:.2g}, n = {len(j)})')

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(j.val_gain, j.test_gain, s=12, alpha=0.5)
ax.axhline(0, color='grey', lw=0.8); ax.axvline(0, color='grey', lw=0.8)
ax.set_xlabel('gain on validation'); ax.set_ylabel('gain on test')
ax.set_title('Does a repair that helped recently help next?')
plt.tight_layout(); plt.show()

## 7. Compare correction policies

Every policy is a **replay** over the tensor: no model is refitted.

In [ ]:
from agents.policies import (NeverCorrectPolicy, AlwaysCorrectPolicy,
                             RandomCorrectPolicy, RGCPolicy, ValidationEvidence)
from agents.reliability import ReliabilityEstimator
from benchmark.oracle import compute_policy_regret

val_labels = add_oracle_labels(outcomes[outcomes.split == 'val'], delta=0.10)
val_sig = signals[signals.instance_id.isin(val_labels.instance_id)]
test_sig = signals[signals.instance_id.isin(labels.instance_id)]

est = ReliabilityEstimator().fit(val_sig, val_labels)
gain = dict(zip(test_sig.instance_id, est.predict_gain(test_sig)))
risk = dict(zip(test_sig.instance_id, est.predict_risk(test_sig)))
thr = ReliabilityEstimator.threshold_for_rate(est.predict_gain(val_sig), 0.30)

evidence = ValidationEvidence(outcomes, 'mase')
rows = []
for policy in [NeverCorrectPolicy(), AlwaysCorrectPolicy(),
               RandomCorrectPolicy(rate=0.30),
               RGCPolicy(gain_threshold=thr, gain_scores=gain, risk_scores=risk)]:
    dec = policy.decide(test_sig, evidence, seed=0)
    sc = compute_policy_regret(outcomes[outcomes.split == 'test'],
                               dec.set_index('instance_id')['chosen_action'], labels)
    corrected = sc[sc.corrected]
    rows.append({
        'policy': policy.name,
        'MASE': sc.chosen_metric.mean(),
        'regret': sc.regret.mean(),
        'correction_rate': sc.corrected.mean(),
        'helped': corrected.improved.mean() if len(corrected) else np.nan,
        'harmed': corrected.harmed.mean() if len(corrected) else np.nan,
        'cost': dec.compute_cost.mean(),
    })
pd.DataFrame(rows).sort_values('MASE').round(4)

**This sample is far too small for inference** -- it exists to show the mechanism.
For real results run:

```bash
python run_experiment.py --config configs/experiment.yaml
python scripts/make_paper_assets.py --run latest
```